# Detecção de Anomalias em ECG com Autocodificadores

## Introdução

Neste laboratório,será analisado um caso de uso dos autocodificadores para detectar anomalias, considerando um cenário em que a base de dados conta com muitas amostras de casos normais e poucos dados de com amostras de caso de falha falha. Ao ter os dados nessa condição, uma gama de soluções aplicáveis para identificar anomalias podem ficar inviáveis de serem exploradas.

Será utilizado um conjunto de dados de sinais de ECG contendo registros rotulados como normais e anormais. O conjunto de dados é composto majoritariamente por traçados cardíacos considerados como saudáveis, que servem como base para o treinamento do autocodificador. Esses sinais apresentam morfologia regular e baixa variabilidade, permitindo que o modelo aprenda um padrão consistente de funcionamento cardíaco. Já durante a fase de teste, são incluídos sinais contendo diferentes tipos de arritmias, caracterizados por distorções, irregularidades ou alterações súbidtas no traçado. Com isso, esse arranjo do conjunto de dados, com muitos dados normais e poucos anormais, torna-o adequado para o cenário de detecção de anomalias, no qual o modelo deve reconstruir bem apenas os padrões saudáveis e apresentar um erro de reconstrução elevado quando exposto a sinais atípicos [<a href="#ref1">1</a>, <a href="#ref2">2</a>]

Em resumo, os seguintes passos serão aplicados:

1. Treinamento do modelo apenas com dados normais (saudáveis).
2. O modelo entende os padrões para reconstruir sinais de ECG de corações saudáveis.
3. No teste, será apresentado dados com anomalia (e.g., aritmia).
4. Como o modelo não consegue reconstruir bem os padrões de dados anormais, a hipótese é que ele falhará em reconstruir o sinal corretamente.
5. O erro de reconstrução será alto. Usamos esse erro alto como indicador da presença de anomalia no sinal avaliado.

## Preparação dos Dados

Utilizaramos o dataset ECG5000.

* **Dados de Treino**: Devem conter apenas batimentos normais.
* **Dados de Teste**: Contêm uma mistura de normais e anômalos para validarmos se o sistema funciona.

Como estamos lidando com séries temporais, os dados vão ter o formato `(amostras, 140, 1)`, onde 140 é o tempo (passos) e 1 é o canal (leitura do sensor).

In [ ]:
from tensorflow.keras import layers, models, callbacks, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
import numpy as np
import os

In [ ]:
# Configuração para reprodutibilidade
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
BATCH_SIZE = 128
EPOCHS = 50

In [ ]:
DATA_URL = "http://storage.googleapis.com/download.tensorflow.org/data/ecg.csv"

In [ ]:
def load_ecg_data(data_url=DATA_URL):
    ### carrega os dados de ecg

    dataframe = pd.read_csv(data_url, header=None)
    raw_data = dataframe.values

    # o último elemento é o rótulo
    labels = raw_data[:, -1]
    data = raw_data[:, 0:-1]

    # separar por classe (1.0 é normal e 0.0 é anômalo)
    train_data = data[labels==1]
    anomalous_data = data[labels==0]

    # normalização dos dados, escala de [0, 1]
    scaler = MinMaxScaler()
    train_data = scaler.fit_transform(train_data)
    anomalous_data = scaler.transform(anomalous_data)

    # divisão de treino e teste
    x_train, x_test_normal = train_test_split(train_data, test_size=0.2, 
                                              random_state=SEED)

    x_train = np.expand_dims(x_train, axis=2)
    x_test_normal = np.expand_dims(x_test_normal, axis=2)
    x_test_anomalous = np.expand_dims(anomalous_data, axis=2)

    print(f"Treino (Apenas Normal): {x_train.shape}")
    print(f"Teste (Apenas Normal): {x_test_normal.shape}")
    print(f"Dados Anômalos (Validação Final): {x_test_anomalous.shape}")

    return x_train, x_test_normal, x_test_anomalous

x_train, x_test_normal, x_test_anomalous = load_ecg_data()    

In [ ]:
def plot_ecg_comparison(normal, anomalous):
    # mostra uma comparação entre um dado normal e anômalo

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(normal[0], color="teal", linewidth=2)
    plt.title("Batimento Normal (Saudável)")
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(anomalous[0], color="firebrick", linewidth=2)
    plt.title("Batimento Anômalo (Aritmia/Erro)")
    plt.grid(True, alpha=0.3)

    plt.show()

plot_ecg_comparison(x_test_normal, x_test_anomalous)

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 1:</b><br>
  Por que é estritamente necessário utilizar apenas dados de batimentos "normais" (saudáveis) durante a fase de treinamento, e qual seria o impacto direto na capacidade de detecção do modelo se dados anômalos fossem incluídos nesse conjunto de treino?
</div>

## Arquitetura: Autoencoder Convolucional 1D

Para sinais como ECG, áudio ou sensores de vibração, é utilziado convoluções 1D (`Conv1D`). O kernel desliza apenas sobre o eixo do tempo.

* **Conv1D e Pooling**: Reduzem a resolução temporal (de 140 passos para 35 no gargalo), forçando o modelo a aprender a forma da opnda e não apenas decorar os pontos.
* **SpatialDropout1D**: Diferente do Dropout comum (que desliga neurônios aleatórios), este desliga mapas de características inteiros. Isso força a rede a não depender de um único filtro para detectar características vitais, aumentando a robustez.
* **Gargalo (*bottleneck*)**: A representação comprimida do que é um "batimento normal".

In [ ]:
def build_ecg_autoencoder(input_shape=(140, 1)):
    # cria um modelo de autoencoder que aprende a reconstruir sinal de ecg saudável.
    
    inputs = layers.Input(shape=input_shape)

    # codificador
    x = layers.Conv1D(32, 7, padding="same", strides=1, use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.MaxPooling1D(pool_size=2, padding="same")(x)
    x = layers.SpatialDropout1D(0.1)(x) 

    x = layers.Conv1D(64, 7, padding="same", strides=1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.MaxPooling1D(pool_size=2, padding="same")(x) 
    
    # espaço latente
    encoded = layers.Conv1D(16, 7, padding="same", strides=1, 
                            kernel_regularizer=regularizers.l2(1e-4),
                            name="bottleneck")(x)
    
    # decodificador
    x = layers.Conv1DTranspose(64, 7, padding="same", strides=2, use_bias=False)(encoded)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    
    x = layers.Conv1DTranspose(32, 7, padding="same", strides=2, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    
    # saída
    outputs = layers.Conv1D(1, 7, activation="sigmoid", padding="same", name="reconstruction")(x)

    model = models.Model(inputs, outputs, name="ecg_anomaly_detector")
    
    return model

model = build_ecg_autoencoder()
model.summary()

In [ ]:
model.compile(optimizer="adam", loss="mae")

In [ ]:
MODEL_DIR = "saved_models"
MODEL_NAME = "ecg_anomaly.keras"
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_NAME)

os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
callbacks_list = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint("saved_models/ecg_anomaly.keras", save_best_only=True),
    callbacks.ReduceLROnPlateau(patience=3)
]

history = model.fit(
    x_train, x_train, epochs=EPOCHS,
    batch_size=BATCH_SIZE, validation_data=(x_test_normal, x_test_normal),
    callbacks=callbacks_list, shuffle=True
)

In [ ]:
# model = tf.keras.models.load_model(MODEL_PATH)

## Definindo o Limiar de Corte (Threshold)

O autocodificador não dá uma classificação exata para anomalias. Ele nos dá um vamor de erro (MAE - *Mean Absolute Error*). É necessário definir estatisticamente o que é um erro alto demais.

Será usado uma abordagem estatística baseada na distribuição dos erros nos dados de treino (que sabemos serem normais):

$$
Threshold = \mu + 2\sigma
$$

Onde $\mu$ é a média do erro e $\sigma$ é o desvio padrão.

* Qualquer novo dado cujo erro de reconstrução ultrapasse esse valor será classificado como anomalia.
* A escolha de $2\sigma$ teoricamente cobre ~95% dos dados normais, deixando os 5% mais externos (possíveis erros) de fora. Em sistemas críticos, esse valor pode ser ajustado para alterar a sensibilidade.

In [ ]:
def plot_reconstruction_error_distribution(model, normal_data, anomalous_data):
    # monta um histograma de erro da distribuição dos dados.

    # predições
    rec_normal = model.predict(normal_data, verbose=0)
    rec_anomalous = model.predict(anomalous_data, verbose=0)

    # calcula o  erro MAE
    mae_normal = np.mean(np.abs(rec_normal - normal_data), axis=1)
    mae_anomalous = np.mean(np.abs(rec_anomalous - anomalous_data), axis=1)

    plt.figure(figsize=(12, 6))
    plt.hist(mae_normal, bins=50, alpha=0.7, color="teal", label="Normais (Treino)")
    plt.hist(mae_anomalous, bins=50, alpha=0.7, color="firebrick", label="Anomalias (Nunca vistos)")

    # definindo uma zona de limiar com métricas estatísticas
    threshold = np.mean(mae_normal) + 2 * np.std(mae_normal)
    plt.axvline(threshold, color="k", linestyle="--", linewidth=2, label=f"Threshold: {threshold:.3f}")

    plt.title("Histograma de erro de reconstrução", fontsize=14)
    plt.xlabel("Erro de reconstrução (MAE)")
    plt.ylabel("Quantidade de Amostras")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    return threshold

calculated_threshold = plot_reconstruction_error_distribution(model, x_test_normal, x_test_anomalous)

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 2:</b><br>
  Considerando que o Autoencoder não fornece uma classificação binária (0 ou 1) por padrão, como o Erro Médio Absoluto (MAE) é utilizado para estabelecer o limiar (threshold) de anomalia, e quais são as consequências práticas de se alterar o multiplicador do desvio padrão (atualmente 2$\sigma$) para um valor maior ou menor?
</div>



## Análise dos Resultados

Nos gráficos a seguir, visualizaremos o funcionamento do detector:

* **Linha Azul**: O sinal real (Entrada).
* **Linha Vermelha**: A tentativa do autoencoder de reconstruí-lo.
* **Área Vermelha**: O erro (diferença absoluta).

Desse modo, podemos esperar os seguintes resultados:

1. **Casos Normais**: Alinha vermelha segue o azul quase perfeitamente. O erro é baixo.
2. **Casos Anômalos**: O modelo tenta desenha um batimento normal por cima de um batimento anômalo. Como as formas não batem (e.g. uma queda onde deveria haver um pico), a área de erro aumenta, o que faz o *threshold* apresentar um valor maior.

In [ ]:
def visualize_anomaly_detection(model, data, threshold, title="Exemplo"):
    # mostra o sinal original em comparação com o sinal reconstruído
    
    reconstruction = model.predict(data[np.newaxis, ...], verbose=0)[0]
    loss = np.mean(np.abs(reconstruction - data))

    plt.figure(figsize=(10,4))
    plt.plot(data, "b", label="Sinal original", linewidth=2)
    plt.plot(reconstruction, "r", label="Reconstrução", linewidth=2)
    plt.fill_between(np.arange(140), data.flatten(), reconstruction.flatten(), color="lightcoral", alpha=0.5, label="Erro")

    status = "Anomalia detectada"if loss > threshold else "Normal"
    color_status = "red" if loss > threshold else "green"

    plt.title(f"{title} | Loss: {loss:.4f} (Limit: {threshold:.4f}) -> {status}", color=color_status, fontweight="bold")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
for i in range(3):
    visualize_anomaly_detection(model, x_test_normal[i], calculated_threshold, title=f"Normal {i}")

In [ ]:
for i in range(3):
    visualize_anomaly_detection(model, x_test_anomalous[i], calculated_threshold, title=f"Anômalo {i}")

## Referências

<p id="ref1">[1] Chen, Z., Yeo, C. K., Lee, B. S., & Lau, C. T. (2018, April). Autoencoder-based network anomaly detection. In 2018 Wireless telecommunications symposium (WTS) (pp. 1-5). IEEE.
<p id="ref2">[2] An, J., & Cho, S. (2015). Variational autoencoder based anomaly detection using reconstruction probability. Special lecture on IE, 2(1), 1-18.